In [105]:
# reading the retail data set
retail <- read.csv("OnlineRetail.csv")

In [106]:
str(retail)

'data.frame':	541909 obs. of  8 variables:
 $ InvoiceNo  : chr  "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr  "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr  "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : int  6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: chr  "12/1/2010 8:26" "12/1/2010 8:26" "12/1/2010 8:26" "12/1/2010 8:26" ...
 $ UnitPrice  : num  2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : int  17850 17850 17850 17850 17850 17850 17850 17850 17850 13047 ...
 $ Country    : chr  "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...


In [120]:
df <- data.frame(retail)
unique(df$Country)

[1] "United Kingdom"       "France"               "Australia"           
 [4] "Netherlands"          "Germany"              "Norway"              
 [7] "EIRE"                 "Switzerland"          "Spain"               
[10] "Poland"               "Portugal"             "Italy"               
[13] "Belgium"              "Lithuania"            "Japan"               
[16] "Iceland"              "Channel Islands"      "Denmark"             
[19] "Cyprus"               "Sweden"               "Austria"             
[22] "Israel"               "Finland"              "Bahrain"             
[25] "Greece"               "Hong Kong"            "Singapore"           
[28] "Lebanon"              "United Arab Emirates" "Saudi Arabia"        
[31] "Czech Republic"       "Canada"               "Unspecified"         
[34] "Brazil"               "USA"                  "European Community"  
[37] "Malta"                "RSA"

## Data Cleaning Plan

- Handle missing Customer IDs:
  - Replace with `"Unknown"`

In [107]:
any(is.na(df$CustomerID)) # I am checking this because I know there is null, I loaded this csv into local database and when selecting "CustomerID" as a PK it throwed error
df$CustomerID[is.na(df$CustomerID)] <- "Unknown"
any(is.na(df$CustomerID)) # any is used to determine if at least one element withthin a logical vector evulates to TRUE

[1] FALSE

[1] FALSE

In [108]:
any(is.na(df)) # We have no missing values in our data frame

[1] TRUE

## Abbreviation
    . We count manually replace every country but it will take time, So I looked up for library.
    . Found a library called "countrycode"
    . In our data like  RSA Republic of South Africa are there let's clearn that up, so that library does not throw warning
    . Here few data are of Islands, let's group that to different category, European Community, Channel Islands and Unspecified to other categories
    . On one side we have filtered based on known name of country "specified_country" and in other where there is islands and another stuff we added them into seprate data frame "specified_country"
    . H

In [109]:
library(countrycode)
library(dplyr) 
# for filter, %in% check element of vector a are present in vector b, it reutns a boolean vector

df$Country[df$Country == "RSA"] <- "South Africa"
df$Country[df$Country == "EIRE"] <- "Ireland"

# how do we know that, well from error using countrycode
unspecified_country <- dplyr::filter(df,  Country %in% c("European Community", "Channel Islands", "Unspecified"))

specified_country <-  dplyr::filter(df,  !Country %in% c("European Community", "Channel Islands", "Unspecified"))

# "country.name" Full English country name, destination is the format of our output, ISO 3-letter code
country_codes <- countrycode(specified_country$Country, origin = "country.name", destination = "iso3c") 

# let's update the country name in "specified_country" 
specified_country$Country <- country_codes

df <- rbind(specified_country, unspecified_country) # as the name says it's binding the rows with came dim cols

## Let's check for the quantity column
    . Here we have negative value, physically item sold cannot be negative, so we are taking the absolute value
    . I looked up for the entry to be exact zero but found none.

In [110]:
df$Quantity <- abs(df$Quantity) # assuming it a type and conversting to a absolute value 
paste("Quantity with exact zero: ",  sum(df[df$Quantity == 0]), " Quantity with non negative: ", sum(df$Quantity)) # we expect larger number than the dim of df, cause more than one items can be sold

[1] "Quantity with exact zero:  0  Quantity with non negative:  5865950"

## Checking for StockCode

In [111]:
any(is.na(df$StockCode)) # We don't have any nulls in stock code.
any(df$StockCode <= 0)

[1] FALSE

[1] FALSE

## Unit price error check
    . 2515 items have UnitPrice of zero.
    . We can find similar product and add the price.
    . If not then we need to drop the rows because its less significant compared to the size of data.
    . Here two of the `UnitPrice` came to be negative, `Description` was `Adjust bad dept` so we are dropping that as well.

In [112]:
any(df[df$UnitPrice < 0])
df <- df[df$UnitPrice > 0, ]
dim(df)

[1] FALSE

[1] 539392      8

## Product Description

In [113]:
check <- df[df$Description == "", ]
any(is.na(df$Description))
# I don't think we have an missing value over there

[1] FALSE

## Working with date 
    . From raw data, dates are given to us as a character, so we are converting it to date object.

In [117]:
df$InvoiceDate <- as.Date(df$InvoiceDate, format = "%m/%d/%Y") # %m represents month in decimal, %m represents month in decimal and %y represents the year 
str(df) # if we look at the str we should see the Date object, 

'data.frame':	539392 obs. of  8 variables:
 $ InvoiceNo  : chr  "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr  "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr  "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : int  6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: Date, format: "2010-12-01" "2010-12-01" ...
 $ UnitPrice  : num  2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : chr  "17850" "17850" "17850" "17850" ...
 $ Country    : chr  NA NA NA NA ...


## Exploratory Data Analysis

In [118]:
summary(df)
unique(df$Country)

  InvoiceNo          StockCode         Description           Quantity       
 Length:539392      Length:539392      Length:539392      Min.   :    1.00  
 Class :character   Class :character   Class :character   1st Qu.:    1.00  
 Mode  :character   Mode  :character   Mode  :character   Median :    3.00  
                                                          Mean   :   10.88  
                                                          3rd Qu.:   10.00  
                                                          Max.   :80995.00  
  InvoiceDate           UnitPrice         CustomerID          Country         
 Min.   :2010-12-01   Min.   :    0.00   Length:539392      Length:539392     
 1st Qu.:2011-03-28   1st Qu.:    1.25   Class :character   Class :character  
 Median :2011-07-20   Median :    2.08   Mode  :character   Mode  :character  
 Mean   :2011-07-04   Mean   :    4.67                                        
 3rd Qu.:2011-10-19   3rd Qu.:    4.13                            

[1] NA                   "USA"                "Channel Islands"   
[4] "Unspecified"        "European Community"